# ICU Mortality Prediction — Machine Learning Pipeline

This **self-contained** Python notebook builds and evaluates predictive models for
in-hospital mortality using OMOP CDM data queried via `sql_query()`.

Steps:

1. Cohort extraction (SQL)
2. Feature engineering (SQL + Python)
3. Data preparation & cleaning
4. Train/test split
5. Logistic regression (sklearn)
6. Decision tree (sklearn)
7. Model comparison & evaluation
8. Calibration analysis
9. Feature importance
10. Threshold analysis & clinical interpretation

**Requirements:** An active database connection with OMOP CDM tables
(person, visit_occurrence, death, measurement, concept).

> `sql_query(sql)` is automatically available in Linkr notebooks.
> It queries the active DuckDB connection and returns a pandas DataFrame.
> Usage: `df = await sql_query("SELECT * FROM person LIMIT 10")`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)
np.random.seed(42)

---
## 1. Cohort Extraction

We create the cohort entirely via SQL — eligible visits (>= 1 day stay),
mortality status, and demographics.

In [ ]:
# Eligible visits: stays >= 1 day
await sql_query("""
    CREATE OR REPLACE VIEW eligible_visits AS
    SELECT
        v.visit_occurrence_id, v.person_id,
        v.visit_start_date,
        v.visit_start_datetime::TIMESTAMP AS visit_start_datetime,
        v.visit_end_date,
        v.visit_end_datetime::TIMESTAMP AS visit_end_datetime,
        EXTRACT(EPOCH FROM (v.visit_end_datetime::TIMESTAMP
            - v.visit_start_datetime::TIMESTAMP)) / 86400.0 AS los_days
    FROM visit_occurrence v
    WHERE EXTRACT(EPOCH FROM (v.visit_end_datetime::TIMESTAMP
        - v.visit_start_datetime::TIMESTAMP)) / 86400.0 >= 1
""")

# In-hospital mortality
await sql_query("""
    CREATE OR REPLACE VIEW visit_mortality AS
    SELECT ev.*,
        CASE WHEN d.death_date IS NOT NULL
             AND d.death_date BETWEEN ev.visit_start_date
                                   AND ev.visit_end_date + INTERVAL '1 day'
             THEN 1 ELSE 0 END AS in_hospital_death
    FROM eligible_visits ev
    LEFT JOIN death d ON ev.person_id = d.person_id
""")

# Final cohort with demographics
await sql_query("""
    CREATE OR REPLACE VIEW cohort AS
    SELECT vm.visit_occurrence_id, vm.person_id, vm.visit_start_datetime,
        EXTRACT(YEAR FROM vm.visit_start_date) - p.year_of_birth AS age,
        p.gender_source_value AS sex, vm.los_days, vm.in_hospital_death
    FROM visit_mortality vm
    JOIN person p ON vm.person_id = p.person_id
    WHERE EXISTS (
        SELECT 1 FROM measurement m
        WHERE m.visit_occurrence_id = vm.visit_occurrence_id
          AND m.value_as_number IS NOT NULL
          AND m.measurement_datetime::TIMESTAMP >= vm.visit_start_datetime
          AND m.measurement_datetime::TIMESTAMP
              <= vm.visit_start_datetime + INTERVAL '24 hours'
    )
""")

In [ ]:
summary = await sql_query("""
    SELECT COUNT(*) AS n_visits, COUNT(DISTINCT person_id) AS n_patients,
        SUM(in_hospital_death) AS n_deaths,
        ROUND(100.0 * SUM(in_hospital_death) / COUNT(*), 1) AS mortality_pct,
        ROUND(AVG(age), 1) AS mean_age,
        ROUND(AVG(los_days), 1) AS mean_los_days
    FROM cohort
""")
print("Cohort:")
summary

---
## 2. Feature Engineering

Extract vital signs, laboratory values, and GCS from the first 24 hours,
then aggregate into a wide-format feature matrix.

| Category | Aggregation |
|---|---|
| Vitals (7) | mean, min, max |
| Labs (15) | first value |
| Neuro / GCS (3) | minimum (worst) |

In [ ]:
# Concept IDs -> column prefixes
VITALS = {
    3027018: "hr",           # Heart rate
    3004249: "sbp",          # Systolic blood pressure
    3012888: "dbp",          # Diastolic blood pressure
    3027598: "mbp",          # Mean blood pressure
    3024171: "resp_rate",    # Respiratory rate
    40762499: "spo2",        # SpO2
    3020891: "temp",         # Temperature
}
LABS = {
    3000963: "hemoglobin",   3023314: "hematocrit",  3024929: "platelets",
    3003282: "wbc",          3019550: "sodium",      3023103: "potassium",
    3014576: "chloride",     3016293: "bicarbonate", 3016723: "creatinine",
    3013682: "bun",          3004501: "glucose",     3037278: "anion_gap",
    3015377: "calcium",      3012095: "magnesium",   3011904: "phosphate",
}
NEURO = {3016335: "gcs_eye", 3009094: "gcs_verbal", 3008223: "gcs_motor"}
ALL_MEASUREMENTS = {**VITALS, **LABS, **NEURO}

In [ ]:
# Extract all measurements in H0-H24
concept_ids = ", ".join(str(cid) for cid in ALL_MEASUREMENTS.keys())
measurements_h24 = await sql_query(f"""
    SELECT m.visit_occurrence_id, m.measurement_concept_id,
           m.value_as_number,
           m.measurement_datetime::TIMESTAMP AS measurement_datetime,
           c.visit_start_datetime
    FROM measurement m
    JOIN cohort c ON m.visit_occurrence_id = c.visit_occurrence_id
    WHERE m.measurement_concept_id IN ({concept_ids})
      AND m.value_as_number IS NOT NULL
      AND m.measurement_datetime::TIMESTAMP >= c.visit_start_datetime
      AND m.measurement_datetime::TIMESTAMP
          <= c.visit_start_datetime + INTERVAL '24 hours'
""")
print(f"Measurements in H0-H24: {len(measurements_h24)} rows")

In [ ]:
# Aggregate per visit x measurement
measurements_h24["feature"] = measurements_h24["measurement_concept_id"].map(ALL_MEASUREMENTS)
vitals_ids, labs_ids, neuro_ids = set(VITALS), set(LABS), set(NEURO)

aggregated_rows = []
for (visit_id, feature), group in measurements_h24.groupby(
    ["visit_occurrence_id", "feature"]
):
    cid = group["measurement_concept_id"].iloc[0]
    values = group.sort_values("measurement_datetime")["value_as_number"]
    if cid in vitals_ids:
        for agg, fn in [("mean", values.mean), ("min", values.min), ("max", values.max)]:
            aggregated_rows.append({"visit_occurrence_id": visit_id,
                "col": f"{feature}_{agg}", "val": fn()})
    elif cid in neuro_ids:
        aggregated_rows.append({"visit_occurrence_id": visit_id,
            "col": f"{feature}_min", "val": values.min()})
    elif cid in labs_ids:
        aggregated_rows.append({"visit_occurrence_id": visit_id,
            "col": f"{feature}_first", "val": values.iloc[0]})

agg_df = pd.DataFrame(aggregated_rows)
print(f"Aggregated features: {len(agg_df)} rows, "
      f"{agg_df['col'].nunique()} distinct columns")

In [ ]:
# Pivot to wide format and merge with cohort demographics
wide = agg_df.pivot_table(
    index="visit_occurrence_id", columns="col",
    values="val", aggfunc="first"
).reset_index()

cohort_df = await sql_query("""
    SELECT visit_occurrence_id, person_id, age, sex, los_days, in_hospital_death
    FROM cohort
""")

df = cohort_df.merge(wide, on="visit_occurrence_id", how="left")

# Sort columns
id_cols = ["visit_occurrence_id", "person_id"]
demo_cols = ["age", "sex", "los_days"]
outcome_cols = ["in_hospital_death"]
feature_cols = sorted([c for c in df.columns
    if c not in id_cols + demo_cols + outcome_cols])
df = df[id_cols + demo_cols + outcome_cols + feature_cols]

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Deaths: {int(df['in_hospital_death'].sum())} / {len(df)} "
      f"({100 * df['in_hospital_death'].mean():.1f}%)")

---
## 3. Data Preparation

In [ ]:
# Feature selection: exclude features with > 30% missing
id_cols = ["visit_occurrence_id", "person_id"]
target = "in_hospital_death"
exclude = id_cols + [target, "sex", "los_days"]

feature_cols = [c for c in df.columns
    if c not in exclude and df[c].dtype in ['float64', 'int64', 'float32']]

missing_pct = df[feature_cols].isnull().mean()
usable = missing_pct[missing_pct < 0.30].index.tolist()
excluded = missing_pct[missing_pct >= 0.30].index.tolist()

print(f"Total features: {len(feature_cols)}")
print(f"Usable (<30% missing): {len(usable)}")
if excluded:
    print(f"Excluded (>= 30% missing): {', '.join(excluded)}")

In [ ]:
# Build model matrix: sex encoding + median imputation
model_df = df[usable + [target]].copy()
model_df['sex_male'] = (df['sex'] == 'M').astype(int)

for col in usable:
    model_df[col] = model_df[col].fillna(model_df[col].median())

feature_names = usable + ['sex_male']
print(f"Model matrix: {model_df.shape[0]} x {len(feature_names)}")

---
## 4. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df[feature_names]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Train: {len(X_train)} ({100 * y_train.mean():.1f}% mortality)")
print(f"Test:  {len(X_test)} ({100 * y_test.mean():.1f}% mortality)")

In [ ]:
# Compare feature distributions between train and test
print(f"{'Feature':<30} {'Train mean':>12} {'Test mean':>12} {'Diff':>8}")
print("-" * 65)
for col in feature_names[:15]:
    tr = X_train[col].mean()
    te = X_test[col].mean()
    print(f"{col:<30} {tr:>12.2f} {te:>12.2f} {abs(tr - te):>8.2f}")

---
## 5. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Top coefficients
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lr_model.coef_[0],
    'Odds Ratio': np.exp(lr_model.coef_[0])
}).sort_values('Coefficient', key=abs, ascending=False)

print("Model coefficients (top 15 by |coefficient|):")
print(coef_df.head(15).to_string(index=False))

In [ ]:
lr_proba_train = lr_model.predict_proba(X_train)[:, 1]
lr_proba_test  = lr_model.predict_proba(X_test)[:, 1]
lr_pred_test   = (lr_proba_test >= 0.5).astype(int)

---
## 6. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    max_depth=5, min_samples_split=20, random_state=42)
tree_model.fit(X_train, y_train)

tree_proba_test = tree_model.predict_proba(X_test)[:, 1]
tree_pred_test  = (tree_proba_test >= 0.5).astype(int)

print("Decision tree feature importances (top 10):")
imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': tree_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(imp.head(10).to_string(index=False))

---
## 7. Model Comparison

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix

def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'AUC': roc_auc_score(y_true, y_prob),
        'Brier': brier_score_loss(y_true, y_prob),
        'Sensitivity': tp / max(tp + fn, 1),
        'Specificity': tn / max(tn + fp, 1),
        'PPV': tp / max(tp + fp, 1),
        'NPV': tn / max(tn + fn, 1),
        'Accuracy': (tp + tn) / len(y_true),
    }

lr_metrics   = compute_metrics(y_test, lr_pred_test, lr_proba_test)
tree_metrics = compute_metrics(y_test, tree_pred_test, tree_proba_test)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'AUC': [lr_metrics['AUC'], tree_metrics['AUC']],
    'Brier': [lr_metrics['Brier'], tree_metrics['Brier']],
    'Sensitivity': [lr_metrics['Sensitivity'], tree_metrics['Sensitivity']],
    'Specificity': [lr_metrics['Specificity'], tree_metrics['Specificity']],
    'PPV': [lr_metrics['PPV'], tree_metrics['PPV']],
    'NPV': [lr_metrics['NPV'], tree_metrics['NPV']],
    'Accuracy': [lr_metrics['Accuracy'], tree_metrics['Accuracy']],
}).round(3)

print("Model Comparison:")
print(comparison.to_string(index=False))

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC curves
for name, proba, color in [
    ('Logistic Regression', lr_proba_test, '#4C78A8'),
    ('Decision Tree', tree_proba_test, '#E45756'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f'{name} (AUC={auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(fontsize=8)

# Confusion matrix (logistic regression)
cm = confusion_matrix(y_test, lr_pred_test)
im = axes[1].imshow(cm, cmap='Blues', aspect='auto')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Alive', 'Dead'])
axes[1].set_yticklabels(['Alive', 'Dead'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (LR)')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                     fontsize=16, fontweight='bold',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

---
## 8. Calibration

In [ ]:
# Calibration: observed vs predicted in bins
def calibration_data(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.digitize(y_prob, bins) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)
    pred_mean = [y_prob[bin_idx == i].mean() if (bin_idx == i).any() else np.nan
                 for i in range(n_bins)]
    obs_mean = [y_true.values[bin_idx == i].mean() if (bin_idx == i).any() else np.nan
                for i in range(n_bins)]
    return np.array(pred_mean), np.array(obs_mean)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, (proba, name, color) in enumerate([
    (lr_proba_test, 'Logistic Regression', '#4C78A8'),
    (tree_proba_test, 'Decision Tree', '#E45756'),
]):
    pred, obs = calibration_data(y_test, proba)
    mask = ~np.isnan(pred)
    axes[i].plot(pred[mask], obs[mask], 'o-', color=color, lw=2, markersize=6)
    axes[i].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[i].set_xlim(0, 1)
    axes[i].set_ylim(0, 1)
    axes[i].set_xlabel('Predicted probability')
    axes[i].set_ylabel('Observed fraction')
    axes[i].set_title(f'Calibration — {name}')

plt.tight_layout()
plt.show()

---
## 9. Feature Importance

In [ ]:
# Logistic regression: standardized coefficients
sds = X_train.std()
std_coefs = lr_model.coef_[0] * sds.values

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'StdCoef': std_coefs,
    'AbsStdCoef': np.abs(std_coefs)
}).sort_values('AbsStdCoef', ascending=False)

top_n = min(20, len(importance_df))
top = importance_df.head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Standardized coefficients
colors_lr = ['#E45756' if v > 0 else '#4C78A8' for v in top['StdCoef']]
axes[0].barh(range(top_n), top['StdCoef'].values, color=colors_lr, edgecolor='white')
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top['Feature'].values, fontsize=8)
axes[0].set_xlabel('Standardized coefficient')
axes[0].set_title('LR: Standardized Coefficients')
axes[0].axvline(x=0, color='black', linewidth=0.5, linestyle='--')
axes[0].invert_yaxis()

# Decision tree: variable importance
tree_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': tree_model.feature_importances_
}).sort_values('Importance', ascending=False)
tree_top = tree_imp_df[tree_imp_df['Importance'] > 0].head(top_n)

if len(tree_top) > 0:
    axes[1].barh(range(len(tree_top)), tree_top['Importance'].values,
                 color='#72B7B2', edgecolor='white')
    axes[1].set_yticks(range(len(tree_top)))
    axes[1].set_yticklabels(tree_top['Feature'].values, fontsize=8)
    axes[1].set_xlabel('Importance')
    axes[1].set_title('Decision Tree: Variable Importance')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 10. Threshold Analysis

In [ ]:
# Evaluate metrics across thresholds
thresholds = np.arange(0.05, 0.96, 0.05)
results = {'Threshold': [], 'Sensitivity': [], 'Specificity': [],
           'PPV': [], 'F1': []}

for t in thresholds:
    pred = (lr_proba_test >= t).astype(int)
    tp = ((pred == 1) & (y_test == 1)).sum()
    tn = ((pred == 0) & (y_test == 0)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    ppv  = tp / max(tp + fp, 1)
    f1   = 2 * sens * ppv / max(sens + ppv, 1e-9)
    results['Threshold'].append(t)
    results['Sensitivity'].append(sens)
    results['Specificity'].append(spec)
    results['PPV'].append(ppv)
    results['F1'].append(f1)

results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, results_df['Sensitivity'], color='#4C78A8', lw=2, label='Sensitivity')
ax.plot(thresholds, results_df['Specificity'], color='#E45756', lw=2, label='Specificity')
ax.plot(thresholds, results_df['PPV'], color='#72B7B2', lw=2, label='PPV')
ax.plot(thresholds, results_df['F1'], color='#F58518', lw=2, ls='--', label='F1')
ax.set_xlabel('Threshold')
ax.set_ylabel('Metric')
ax.set_title('Threshold Analysis (Logistic Regression)')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

# Optimal threshold (max F1)
best_idx = results_df['F1'].idxmax()
best = results_df.iloc[best_idx]
print(f"\nOptimal threshold (max F1): {best['Threshold']:.2f}")
print(f"  Sensitivity: {100 * best['Sensitivity']:.1f}%")
print(f"  Specificity: {100 * best['Specificity']:.1f}%")
print(f"  PPV:         {100 * best['PPV']:.1f}%")
print(f"  F1:          {best['F1']:.3f}")

---
## 11. Summary

In [ ]:
print("=" * 70)
print("STUDY SUMMARY — IN-HOSPITAL MORTALITY PREDICTION")
print("=" * 70)

print(f"""
COHORT
  Visits:     {len(df)}
  Patients:   {df['person_id'].nunique()}
  Deaths:     {int(df['in_hospital_death'].sum())} ({100 * df['in_hospital_death'].mean():.1f}%)
  Age:        {df['age'].mean():.0f} +/- {df['age'].std():.0f} years

DATA
  Features:   {len(feature_names)} ({len(usable)} clinical + sex)
  Train/Test: {len(X_train)} / {len(X_test)} (75/25 stratified split)
  Missing:    Median imputation (features with >30% missing excluded)

RESULTS
  Logistic Regression
    AUC:  {lr_metrics['AUC']:.3f}  |  Brier: {lr_metrics['Brier']:.3f}
    Sens: {100 * lr_metrics['Sensitivity']:.1f}%  |  Spec: {100 * lr_metrics['Specificity']:.1f}%

  Decision Tree
    AUC:  {tree_metrics['AUC']:.3f}  |  Brier: {tree_metrics['Brier']:.3f}
    Sens: {100 * tree_metrics['Sensitivity']:.1f}%  |  Spec: {100 * tree_metrics['Specificity']:.1f}%

TOP PREDICTORS (by |standardized coefficient|)""")

for _, row in importance_df.head(10).iterrows():
    print(f"  {row['Feature']:<30} {row['StdCoef']:+.4f}")

print("""
LIMITATIONS
  - Single-center data (MIMIC-IV, Beth Israel Deaconess)
  - H0-H24 measurements only (early prediction window)
  - Median imputation for missing values
  - No external validation cohort
  - No temporal validation (train on earlier, test on later)
  - Potential survivorship bias (patients who die within 24h excluded)

NEXT STEPS
  - External validation on eICU or another OMOP CDM database
  - Temporal validation split
  - Multiple imputation instead of median
  - Additional features: diagnoses, procedures, medications
  - Ensemble methods (random forest, gradient boosting)
  - Prospective evaluation
""")